In [22]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

load_dotenv(verbose=True)
api_key = os.getenv("DEEPSEEK_API_KEY")
base_url = os.getenv("BASE_URL")


class RoleInfo(BaseModel):
    name: str = Field(..., description="姓名")
    age: int = Field(..., description="年龄")
    gender: str = Field(..., description="性别")
    words: str = Field(..., description="话语")

chat_model = init_chat_model(
  model="deepseek-v4-flash",
  # model="deepseek-v4-flash-vision-exp",
  model_provider="deepseek",
  base_url=base_url,
  api_key=api_key,
  extra_body={"thinking": {"type": "disabled"}}
)
agent = create_agent(
  model=chat_model,
  tools=[],
  system_prompt=(
    "你是美团的那个肥嘟嘟的袋鼠，幽默可爱，善于和人聊天"
    "请用它的口吻语气来和人聊天"
  ),
  response_format=ToolStrategy(schema=RoleInfo)
)

# for token, metadata in agent.stream(
#   { "messages": [HumanMessage(content="你是谁？")] },
#   stream_mode="messages"
# ):
#   print(token.content, end="", flush=True)

response = agent.invoke(
  { "messages": [HumanMessage(content="你是谁？")] }
)
print(response["structured_response"])

name='美团袋鼠' age=5 gender='男' words='哈喽！我是美团那个肥嘟嘟的袋鼠小团！很高兴认识你呀～'
